# Create engine-disjoint protocols for FD002 and FD003.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
import hashlib,json,platform
import numpy as np
import pandas as pd


AUDIT_DIR=Path("revision_outputs")/"08_FD002_FD003_data_audit"
OUTPUT_DIR=Path("revision_outputs")/"09_FD002_FD003_protocol"
DATASETS=("FD002","FD003")
SPLIT_SEED=42
MODEL_SEEDS=(42,43,44)
N_FOLDS=10
RUL_WINDOWS=(20,30,40)
PRIMARY_RUL=30
SENSORS=[f"Sensor{i}" for i in range(1,22)]

APPROVED_SOURCES={'FD002': {'rows': 53759, 'units': 260, 'train_raw_sha256': 'dac6c4dbc4e7c1bdeb5747da3d313d05c395bb99801b44a002b26a2ba13d788f', 'train_metadata_sha256': '31cc7272711151a2be5cb10b9c33925291a8d48cb69380f839f904a5ec0a6bb9'}, 'FD003': {'rows': 24720, 'units': 100, 'train_raw_sha256': '2abbe9968cc5e8eb091980f51b20f62bb4127336d3482cb52071d53bf23329e2', 'train_metadata_sha256': '2ecd0bce075d55652dde45e8f4b57a908a7798a2b362e9b870e5b47042d00bf8'}}


In [ ]:
def require(ok,message):
    if not bool(ok):raise ValueError(message)

def digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def canonical(value):return json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(",",":"),allow_nan=False)

def validate_metadata(m,ds,rows,units):
    cols=['dataset','split','source_row','UnitNumber','Time','engine_key','sample_key','RUL']
    require(set(cols).issubset(m.columns) and len(m)==rows,'Eğitim metadata şeması/satır sayısı farklı.')
    require(m[cols].notna().all().all(),'Metadata içinde eksik değer var.')
    require(m.dataset.eq(ds).all() and m.split.eq('train').all(),'Veri kümesi/bölme karışmış.')
    m=m.copy()
    for col in ['source_row','UnitNumber','Time','RUL']:
        a=pd.to_numeric(m[col],errors='raise').to_numpy(dtype=float)
        require(np.isfinite(a).all() and np.all(a==np.floor(a)),f'{col} sonlu tam sayı olmalı.')
        m[col]=a.astype('int64')
    require(np.array_equal(m.source_row.to_numpy(),np.arange(rows)),'source_row sırası değişmiş.')
    require(np.array_equal(np.sort(m.UnitNumber.unique()),np.arange(1,units+1)),'Motor kimlikleri farklı.')
    require(not m.duplicated(['UnitNumber','Time']).any(),'Tekrarlanan motor/çevrim var.')
    engines=ds+':train:'+m.UnitNumber.astype(str)
    require(m.engine_key.eq(engines).all(),'Motor anahtarı geçersiz.')
    require(m.sample_key.eq(engines+':'+m.Time.astype(str)).all() and m.sample_key.is_unique,'Örnek anahtarı geçersiz.')
    for engine,g in m.groupby('engine_key',sort=False):
        require(np.array_equal(g.Time.to_numpy(),np.arange(1,len(g)+1)),f'{engine}: çevrim sırası bozuk.')
        require(np.array_equal(g.RUL.to_numpy(),g.Time.max()-g.Time.to_numpy()),f'{engine}: RUL hesabı farklı.')
    return m

def stable_order(keys,salt):
    return sorted(keys,key=lambda k:(hashlib.sha256(f'{SPLIT_SEED}|{salt}|{k}'.encode()).hexdigest(),k))

def make_splits(ds,keys):
    keys=sorted(set(keys))
    require(len(keys)%N_FOLDS==0,'Motor sayısı 10 eşit dış fold oluşturmalı.')
    size=len(keys)//N_FOLDS
    groups=np.array_split(np.array(stable_order(keys,'outer'),dtype=object),N_FOLDS)
    rows=[]
    for fold,outer in enumerate(groups,start=1):
        others=stable_order(set(keys)-set(outer.tolist()),f'inner:{fold}')
        parts={'early_stop':others[:size],'selection':others[size:2*size],
               'fit':others[2*size:],'outer_valid':outer.tolist()}
        for role,engines in parts.items():
            rows.extend({'dataset':ds,'scheme':'cv','fold':fold,'engine_key':e,'role':role} for e in engines)
    final=stable_order(keys,'final')
    for role,engines in {'early_stop':final[:size],'selection':final[size:2*size],'fit':final[2*size:]}.items():
        rows.extend({'dataset':ds,'scheme':'final','fold':0,'engine_key':e,'role':role} for e in engines)
    return pd.DataFrame(rows).sort_values(['dataset','scheme','fold','engine_key']).reset_index(drop=True)

def validate_splits(s,ds,keys):
    keys=set(keys);n=len(keys);size=n//10
    require(s.dataset.eq(ds).all() and set(s.scheme)=={'cv','final'},'Bölme kimliği farklı.')
    require(not s.duplicated(['scheme','fold','engine_key']).any(),'Aynı motor birden fazla role atanmış.')
    cv=s.loc[s.scheme.eq('cv')]
    require(set(cv.fold)==set(range(1,11)),'Dış fold kapsamı farklı.')
    for _,g in cv.groupby('fold'):
        require(len(g)==n and set(g.engine_key)==keys,'Bir fold tüm motorları tam bir kez içermeli.')
        require(g.role.value_counts().to_dict()=={'fit':7*size,'early_stop':size,'selection':size,'outer_valid':size},'CV rol sayıları farklı.')
    outer=cv.loc[cv.role.eq('outer_valid'),'engine_key'].value_counts()
    require(set(outer.index)==keys and outer.eq(1).all(),'Her motor tam bir kez dış değerlendirmede olmalı.')
    final=s.loc[s.scheme.eq('final')]
    require(len(final)==n and set(final.engine_key)==keys and final.fold.eq(0).all(),'Nihai bölme kapsamı farklı.')
    require(final.role.value_counts().to_dict()=={'fit':8*size,'early_stop':size,'selection':size},'Nihai rol sayıları farklı.')

def write_once(path,payload):
    if path.exists():require(path.read_bytes()==payload,f'Mevcut çıktı farklı; üzerine yazılmadı: {path}')
    else:
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(payload)

def csv_bytes(frame):return frame.to_csv(index=False,lineterminator='\n').encode('utf-8')

def prepare_inputs(audit_dir,approved):
    summary_path=audit_dir/'additional_data_summary.json'
    require(summary_path.is_file(),f'08 özeti yok: {summary_path}')
    a=json.loads(summary_path.read_text(encoding='utf-8'))
    require(a.get('stage')=='08_additional_data_audit' and a.get('status')=='PASS','08 doğrulaması PASS olmalı.')
    require(a.get('model_training_performed') is False and a.get('RUL_in_model_inputs') is False,'08 girdi ayrımı değişmiş.')
    refs={};splits={};role_rows=[];counts={}
    for ds in DATASETS:
        info=a['datasets'][ds];expected=approved[ds]
        require(info['status']=='PASS' and info['train']['shape']==[expected['rows'],26]
                and info['train']['units']==expected['units'],'08 eğitim boyutu farklı.')
        require(info['feature_columns']==SENSORS and info['rows_removed']==0 and not info['sensors_removed'],
                'Sensör/satır kapsamı farklı.')
        require(info['source_raw_file_sha256'][f'train_{ds}.txt']==expected['train_raw_sha256'],'08 ham eğitim dosyası kaydı farklı.')
        relative=ds+'/train_evaluation_metadata.csv'
        path=audit_dir/relative
        require(path.is_file() and digest(path)==expected['train_metadata_sha256']
                and info['output_file_sha256'][relative]==expected['train_metadata_sha256'],
                f'08 eğitim metadatası eksik/değişmiş: {relative}')
        m=validate_metadata(pd.read_csv(path),ds,expected['rows'],expected['units'])
        counts[ds]=[]
        for w in RUL_WINDOWS:
            m[f'late_life_le_{w}']=m.RUL.le(w).astype('int8')
            require(m[f'late_life_le_{w}'].nunique()==2,f'{ds}, RUL {w}: iki sınıf da bulunmalı.')
            counts[ds].append({'RUL_window':w,'positive':int(m[f'late_life_le_{w}'].sum()),
                               'negative':int((1-m[f'late_life_le_{w}']).sum())})
        s=make_splits(ds,m.engine_key.unique())
        validate_splits(s,ds,m.engine_key.unique())
        for (scheme,fold),g in s.groupby(['scheme','fold'],sort=True):
            role=m.engine_key.map(g.set_index('engine_key').role)
            require(role.notna().all(),'Rol atanmamış eğitim satırı var.')
            for name in sorted(g.role.unique()):
                selected=m.loc[role.eq(name)]
                row={'dataset':ds,'scheme':scheme,'fold':int(fold),'role':name,
                     'engines':int(selected.engine_key.nunique()),'rows':len(selected)}
                for w in RUL_WINDOWS:
                    y=selected[f'late_life_le_{w}']
                    require(y.nunique()==2,f'{ds}/{scheme}/{fold}/{name}: referans tek sınıf.')
                    row[f'positive_le_{w}']=int(y.sum())
                    row[f'negative_le_{w}']=int((1-y).sum())
                role_rows.append(row)
        refs[ds]=m;splits[ds]=s
    return refs,pd.concat(list(splits.values()),ignore_index=True),pd.DataFrame(role_rows),counts


In [ ]:
require((DATASETS,SPLIT_SEED,MODEL_SEEDS,N_FOLDS,RUL_WINDOWS,PRIMARY_RUL)==
        (("FD002","FD003"),42,(42,43,44),10,(20,30,40),30),'Sabit protokol ayarlarını değiştirmeyin.')
reference,splits,role_summary,reference_counts=prepare_inputs(AUDIT_DIR,APPROVED_SOURCES)

BLOCKS=[{'dataset':'FD002','preprocessing':'global'},
        {'dataset':'FD002','preprocessing':'conditionwise'},
        {'dataset':'FD003','preprocessing':'global'}]
protocol={
    'protocol_version':'FD002_FD003_reference_splits_v1',
    'source_training_files':APPROVED_SOURCES,
    'split':{'method':'SHA256(seed|salt|engine_key), sorted digest then engine key',
             'seed':42,'outer_salt':'outer','inner_salt':'inner:{fold}','final_salt':'final',
             'outer_folds':10,'cv_motor_fractions':{'fit':.7,'early_stop':.1,'selection':.1,'outer_valid':.1},
             'final_motor_fractions':{'fit':.8,'early_stop':.1,'selection':.1}},
    'model_seeds':[42,43,44],
    'reference':{'primary_RUL_threshold':30,'sensitivity_RUL_thresholds':[20,40],
                 'rule':'RUL <= threshold; RUL=0 included; keep all cycles',
                 'is_anomaly_onset_ground_truth':False,'RUL_is_model_input':False,
                 'sensitivity_refit_or_rethreshold':False},
    'analysis_blocks':BLOCKS,'sensor_inputs':SENSORS,'remove_constant_sensors':False,
    'preprocessing':{
        'global':'StandardScaler on fit engines only; transform held-out roles without refit',
        'conditionwise':{'dataset':'FD002','settings':['OpSet1','OpSet2','OpSet3'],
                         'settings_scaler':'StandardScaler on fit settings only, float64',
                         'KMeans':{'n_clusters':6,'random_state':42,'n_init':20,'max_iter':300,'tol':1e-4,
                                   'algorithm':'lloyd','init':'k-means++'},
                         'sensor_scalers':'one StandardScaler per fit-assigned cluster; 21 sensors',
                         'held_out_assignment':'predict nearest cluster with fixed settings scaler and KMeans',
                         'minimum_fit_rows_per_cluster':2,'require_all_six_fit_clusters':True,
                         'fallback':'raise error, no silent substitution','extra_global_sensor_scaling':False,
                         'settings_are_detector_or_AE_inputs':False},
        'sensor_dtype':'float32 before sensor StandardScaler; transformed AE inputs float32'},
    'representations':['Raw','PCA_d4','PCA_scaled_d4','AE_deep_d21','AE_deep_d4'],
    'AE':{'latent_dimensions':[21,4],'widths_template':[21,42,21,'d',21,42,21],
          'hidden_activation':'relu','latent_activation':'linear','output_activation':'linear',
          'optimizer':'Adam','learning_rate':.001,'loss':'mse','max_epochs':100,'batch_size':256,
          'shuffle':True,'patience':10,'early_stopping_monitor':'val_loss','restore_best_weights':True,
          'validation_role':'early_stop','latent_scaler_fit_role':'fit','deterministic_ops':True},
    'PCA':{'n_components':4,'svd_solver':'full','whiten':False,'dtype':'float64',
           'post_scaling':'fit-only StandardScaler only for PCA_scaled_d4'},
    'detectors':{'IF':{'candidates':[{'n_estimators':n} for n in [100,200,400]],
                        'fixed':{'max_samples':'auto','contamination':'auto','n_jobs':-1},
                        'random_state':'model seed','score':'negative score_samples'},
                 'LOF':{'candidates':[{'n_neighbors':n} for n in [20,50,100]],
                        'fixed':{'novelty':True,'contamination':'auto','metric':'euclidean','n_jobs':-1},
                        'score':'negative novelty score_samples on separate query data'},
                 'kNN':{'candidates':[{'n_neighbors':n} for n in [20,50,100]],
                        'fixed':{'metric':'euclidean','n_jobs':-1},
                        'score':'kth Euclidean distance to fit rows; selection/evaluation queried separately'}},
    'standalone_scores':['AE_deep_d21 reconstruction MSE','AE_deep_d4 reconstruction MSE'],
    'MSE_definition':'mean squared error over 21 branch-normalized sensor inputs',
    'selection':{'role':'selection','primary_metric':'average_precision','reference_RUL':30,
                 'parameter_tie':'first candidate in recorded order',
                 'threshold_metric':'positive F1','threshold_tie':'higher threshold within atol=1e-12, rtol=0',
                 'decision_rule':'score >= threshold','refit_after_selection':False,
                 'is_label_informed':True},
    'reporting':{'primary_metric':'average_precision','secondary_metrics':['ROC_AUC','precision','recall','F1','Macro_F1'],
                 'aggregation':'equal-weight fold metric means within each dataset and preprocessing block',
                 'sd':'descriptive, not independent-replicate confidence interval',
                 'pooled_fold_AP':False,'automatic_significance_test':False,
                 'publish_all_blocks':True,'select_winner_on_outer_folds':False},
    'prior_information':'AE4/PCA4 comparisons specified after FD001 results; additional-dataset performance not yet observed',
    'official_test':'not used in CV fitting, early stopping, selection, or this protocol stage',
    'scope_note':'Three-detector comparison; not completion of every reviewer request or every original manuscript baseline'
}
protocol_sha256=hashlib.sha256(canonical(protocol).encode()).hexdigest()
payloads={'experiment_protocol.json':(json.dumps(protocol,ensure_ascii=False,indent=2,allow_nan=False)+'\n').encode(),
          'engine_splits.csv':csv_bytes(splits),'role_summary.csv':csv_bytes(role_summary)}
for ds,m in reference.items():payloads[ds+'/train_evaluation_reference.csv']=csv_bytes(m)
for name,payload in payloads.items():
    path=OUTPUT_DIR/name
    if path.exists():require(path.read_bytes()==payload,f'Mevcut protokol çıktısı farklı: {path}; üzerine yazılmadı.')
for name,payload in payloads.items():write_once(OUTPUT_DIR/name,payload)

summary={'stage':'09_additional_protocol','status':'PASS','canonical_protocol_sha256':protocol_sha256,
         'protocol_version':protocol['protocol_version'],'datasets':{},'analysis_blocks':BLOCKS,
         'model_seeds':list(MODEL_SEEDS),'outer_folds':10,'primary_RUL_threshold':30,
         'sensitivity_RUL_thresholds':[20,40],'representations':protocol['representations'],
         'detectors':['IF','LOF','kNN'],'test_files_read':False,'raw_sensor_files_read':False,
         'models_trained':False,'learned_preprocessing_performed':False,
         'output_file_sha256':{name:hashlib.sha256(payload).hexdigest() for name,payload in payloads.items()},
         'source_training_files':APPROVED_SOURCES,
         'role_summary':role_summary.to_dict('records')}
for ds,expected in APPROVED_SOURCES.items():
    n=expected['units'];unit=n//10
    summary['datasets'][ds]={'train_rows':expected['rows'],'train_engines':n,
        'cv_engines_per_role':{'fit':7*unit,'early_stop':unit,'selection':unit,'outer_valid':unit},
        'final_engines_per_role':{'fit':8*unit,'early_stop':unit,'selection':unit},
        'each_engine_outer_valid_exactly_once':True,'reference_counts':reference_counts[ds]}
summary_path=OUTPUT_DIR/'additional_protocol_summary.json'
write_once(summary_path,(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False)+'\n').encode())
print(role_summary.loc[role_summary.scheme.eq('cv') & role_summary.fold.eq(1),
                       ['dataset','role','engines','rows','positive_le_30','negative_le_30']].to_string(index=False))
print('ADDITIONAL PROTOCOL: PASS')
print('Gönderilecek dosya:',summary_path.resolve())
